In [1]:
import pandas as pd
from datasets import load_dataset
from collections import defaultdict
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

/Users/mohammed.elkholy/Desktop/Projects/Seg/ReconstructingDocs/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# dataset = load_dataset("CAMeL-Lab/BAREC-Corpus-v1.0")
dataset = pd.read_csv("adjusted_splits.tsv", sep="\t")

In [22]:
dataset.head()

,Unnamed: 0,id,text,source,doc_name,split,paragraph_id,sent_id_in_paragraph,word_count,new_split
0,0,601006001001,كلمة عربية تعني الطوفان :,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,1,5,train
1,1,601006001002,( أ ) الهاريكين,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,2,4,train
2,2,601006001003,( ب ) التورنادو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,3,4,train
3,3,601006001004,( ج ) التيفون,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,4,4,train
4,4,601006001005,( د ) النينو,ArabicMMLU,ArabicMMLU_Group_31_006,train,1,5,4,train


In [5]:
punctuation_common = [
    # single
    ".", ",", "،", ";", "؛", "!", "?", "؟",

    # repeated / emphasis
    "..", "...",
    "!!", "!!!",
    "??", "؟؟",

    # mixed emphasis (common)
    "!?", "?!",
    "؟!", "!؟"
]

form_punct = [
    ",", "(", ")", "-", '"'# "[", "]", 
]
sent_end_punct = [ ".", ",", "؛", "!", "؟"]
sent_end_punct_pattern = re.compile(
    "|".join(
        sorted(
            map(re.escape, sent_end_punct),
            key=len,
            reverse=True
        )
    )
)

def ends_with_pnx(text):
    text = text.split(" ")
    # use this pattern to check if the last word is a pnx r'[^\w\s]'
    pnx_pattern = re.compile(r'[^\w\s]')
    if re.search(pnx_pattern, text[-1]):
        return True
    return False
def cnt_sent_end_punct(text):
    return len(re.findall(sent_end_punct_pattern, text))

def cnt_specific_pnx(text, pnx):
    return text.count(pnx)

def ends_with_specific_pnx(text, pnx):
    return text.endswith(pnx)


def has_sent_end_punct(text):
    return re.search(sent_end_punct_pattern, text) is not None

def ends_with_comma(text):
    return text.endswith(",")

def starts_with_conjunction_word(text, conjunction_word="و"):
    return text.startswith(conjunction_word)

def ends_with_sent_end_punct(text):
    split_text = text.split(' ')
    if split_text[-1] in sent_end_punct:
        return True
    return False

def get_num_common_punct(text):
    common_punct_pattern = re.compile(
        "|".join(
            sorted(
                map(re.escape, punctuation_common),
                key=len,
                reverse=True    
            )
        )
    )
    # count number of common punct in text
    return len(re.findall(common_punct_pattern, text))

def get_num_form_punct(text):
    form_punct_pattern = re.compile(
        "|".join(
            sorted(
                map(re.escape, form_punct),
                key=len,
                reverse=True
            )
        )
    )
    # count number of form punct in text
    return len(re.findall(form_punct_pattern, text))


In [10]:
from collections import defaultdict
src_total_sents = defaultdict(lambda: defaultdict(int))
all_sources = defaultdict(set)
docs_per_src = defaultdict(lambda: defaultdict(set))
n_docs_per_src = defaultdict(lambda: defaultdict(int))

def read_dataset(df):
    data = defaultdict(list)
    for split in df["new_split"].unique():
        print(f"Processing split: {split} with {len(df[df['new_split'] == split])} examples")
        split_df = df[df['new_split'] == split]
        
        for _, row in split_df.iterrows():
            all_sources[split].add(row['source'])
            src_total_sents[split][row['source']] += 1
            docs_per_src[split][row['source']].add(row['doc_name'])
            ex = {
                'id': row['id'],
                'sent': row['text'],
                'source': row['source'],
                'document': row['doc_name'],
                'length': len(row['text'].split(" ")),
                'split': split,
                'has_sent_end_punct': has_sent_end_punct(row['text']),
                'labels': [0] * (len(row["text"].split(" ")) - 1) + [1],
                "ends_with_sent_end_punct": ends_with_sent_end_punct(row['text']),
                "num_common_punct": get_num_common_punct(row['text']),
                "num_form_punct": get_num_form_punct(row['text']),
                "ends_with_comma": ends_with_comma(row['text']),
                "starts_with_conjunction_word": starts_with_conjunction_word(row['text']),
                "ends_with_pnx": ends_with_pnx(row['text']),
            }
            for pnx in sent_end_punct:
                ex[f"ends_with_{pnx}"] = ends_with_specific_pnx(row['text'], pnx)
                ex[f"num_{pnx}"] = cnt_specific_pnx(row['text'], pnx)
            assert len(ex["labels"]) == ex["length"] == len(ex["sent"].split(" ")), f"Length of labels {len(ex['labels'])} does not match length of text {ex['length']}, {len(ex['sent'].split(' '))}"
            data[split].append(ex)
    return data


def read_barec(dataset):
    data = defaultdict(list)
    for split in dataset.keys():
        print(f"Processing split: {split} with {len(dataset[split])} examples")
        for example in dataset[split]:
            all_sources[split].add(example['Source'])
            src_total_sents[split][example['Source']] += 1
            # print(f"Source: {example['Source']}")
            docs_per_src[split][example['Source']].add(example['Document'])
            ex = {'id': example['ID'], 
            'sent': example['Word'], 
            'source': example['Source'], 
            'document': example['Document'], 
            'length': len(example['Word'].split(" ")), 
            "Annotator": example['Annotator'], 
            "R3": example["Readability_Level_3"], 
            "R5": example["Readability_Level_5"], 
            "R7": example["Readability_Level_7"], 
            "R19": example["Readability_Level_19"], "num_sent_end_punct": cnt_sent_end_punct(example['Word']), 
            "has_sent_end_punct": has_sent_end_punct(example['Word']), 
            "split": split, 
            "ends_with_sent_end_punct": ends_with_sent_end_punct(example['Word']), 
            "num_common_punct": get_num_common_punct(example['Word']),
            "num_form_punct": get_num_form_punct(example['Word']),
            "ends_with_comma": ends_with_comma(example['Word']),
            "starts_with_conjunction_word": starts_with_conjunction_word(example['Word']),
            "ends_with_pnx": ends_with_pnx(example['Word']),
            }
            ex["labels"] = [0 for _ in range(ex["length"]-1)] + [1]
            for pnx in sent_end_punct:
                ex[f"ends_with_{pnx}"] = ends_with_specific_pnx(example['Word'], pnx)
                ex[f"num_{pnx}"] = cnt_specific_pnx(example['Word'], pnx)
            assert len(ex["labels"]) == ex["length"] == len(ex["sent"].split(" ")), f"Length of labels {len(ex['labels'])} does not match length of text {ex['length']}, {len(ex['sent'].split(' '))}"
            data[split].append(ex)
            
    return data

# data = read_barec(dataset)
data = read_dataset(dataset)
for split in data:
    for src in all_sources[split]:
        # print(f"Source: {src}")
        # print(docs_per_src[split][src])
        n_docs_per_src[split][src] = len(list(docs_per_src[split][src]))

Processing split: train with 670616 examples
Processing split: test with 137353 examples
Processing split: STask with 5663 examples
Processing split: dev with 136677 examples


In [11]:
# group docs by document, sorted by id before joining on " "
grouped_examples_by_split = defaultdict(list)
for split in data:
    for example in data[split]:
        if example['split'] == split:
            grouped_examples_by_split[split].append(example)
grouped_examples_by_doc = defaultdict(lambda: defaultdict(list))
for split in data:
    for example in data[split]:
        if example['split'] == split:
            grouped_examples_by_doc[split][example['document']].append(example)
def generate_grouped_doc_data(data):
    grouped_doc_data = defaultdict(lambda: defaultdict(list))
    for split in data:
        for doc in data[split]:
            # sort examples in doc by ID
            # print(data[split][doc])
            data[split][doc] = sorted(data[split][doc], key=lambda x: x['id'])
            curr_doc_text = []
            curr_doc_labels = []
            for i in data[split][doc]:
                curr_doc_text.extend(i['sent'].split(" "))
                curr_doc_labels.extend(i['labels'])
            assert len(curr_doc_labels) == len(curr_doc_text), f"Length of labels {len(curr_doc_labels)} does not match length of text {len(curr_doc_text)}"
            curr_doc_source = data[split][doc][0]['source']
            curr_doc_lengths = [i['length'] for i in data[split][doc]]
            # curr_doc_r19 = [i['R19'] for i in data[split][doc]]
            grouped_doc_data[split][doc] = {
                "text": curr_doc_text, 
                "labels": curr_doc_labels, 
                "source": curr_doc_source, 
                "lengths": curr_doc_lengths, 
                # "r19": curr_doc_r19
                }
    return grouped_doc_data

grouped_doc_data = generate_grouped_doc_data(grouped_examples_by_doc)

In [23]:
import re
import bisect

tokens = grouped_doc_data["train"]["BAREC_Majed_0413_1987_002.txt"]["text"]
labels = grouped_doc_data["train"]["BAREC_Majed_0413_1987_002.txt"]["labels"]  # adjust key if needed

pattern = r'(?:\. ){2,}'  # ". . " or more, including trailing space

# 1) Build text + token char spans (start,end) in the joined string
starts = []
ends = []
parts = []
pos = 0

for i, tok in enumerate(tokens):
    if i > 0:
        parts.append(" ")
        pos += 1
    starts.append(pos)
    parts.append(tok)
    pos += len(tok)
    ends.append(pos)

text = "".join(parts)

# 2) Regex over the joined text (char spans)
matches = list(re.finditer(pattern, text))

# For fast mapping: list of token starts for bisect
# token i spans [starts[i], ends[i])
# 3) Convert char span -> token span
token_spans = []
for m in matches:
    s, e = m.start(), m.end()

    # token containing the start char
    i0 = bisect.bisect_right(starts, s) - 1
    if i0 < 0:
        i0 = 0

    # token containing the last matched char (e-1)
    i1 = bisect.bisect_right(starts, e - 1) - 1
    if i1 < 0:
        i1 = 0

    # Optional: tighten to tokens that actually overlap the match
    # (in case match begins/ends in whitespace)
    while i0 < len(tokens) and ends[i0] <= s:
        i0 += 1
    while i1 >= 0 and starts[i1] >= e:
        i1 -= 1

    token_spans.append((i0, i1))  # inclusive end

# 4) Use token spans to index labels
for (i0, i1), m in zip(token_spans, matches):
    print("match:", repr(m.group()), "chars:", (m.start(), m.end()))
    print("tokens:", tokens[i0:i1+1])
    print("labels:", labels[i0:i1+1])
    print()


match: '. . ' chars: (185, 189)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (226, 230)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (241, 245)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (468, 472)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (585, 589)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (668, 672)
tokens: ['.', '.']
labels: [0, 0]

match: '. . ' chars: (707, 711)
tokens: ['.', '.']
labels: [0, 0]

match: '. . ' chars: (736, 740)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (1025, 1029)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (1244, 1248)
tokens: ['.', '.']
labels: [0, 0]

match: '. . ' chars: (1348, 1352)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (1426, 1430)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (1455, 1459)
tokens: ['.', '.']
labels: [0, 1]

match: '. . ' chars: (1562, 1566)
tokens: ['.', '.']
labels: [0, 1]

match: '. . . ' chars: (1600, 1606)
tokens: ['.', 

In [12]:
from collections import defaultdict

cnt_consecutive_periods = defaultdict(lambda: defaultdict(int))
cnt_spans_with_1 = defaultdict(lambda: defaultdict(int))  # NEW
period_spans = defaultdict(lambda: defaultdict(list))

for split in data:
    for doc in grouped_doc_data[split]:
        tokens = grouped_doc_data[split][doc]["text"]
        labels = grouped_doc_data[split][doc]["labels"]

        i = 0
        while i < len(tokens):
            if tokens[i] == ".":
                j = i
                while j < len(tokens) and tokens[j] == ".":
                    j += 1

                run_len = j - i
                if run_len >= 2:
                    start, end = i, j - 1

                    # record span
                    period_spans[split][doc].append((start, end))
                    cnt_consecutive_periods[split][doc] += 1

                    # ✅ check if any label == 1 inside span
                    if any(label == 1 for label in labels[start:end+1]):
                        cnt_spans_with_1[split][doc] += 1

                i = j
            else:
                i += 1
# example access
# print(cnt_consecutive_periods["train"]["BAREC_Majed_0413_1987_002.txt"])
# print(cnt_spans_with_1["train"]["BAREC_Majed_0413_1987_002.txt"])

In [13]:
data["test"][7]

{'id': 601003002003,
 'sent': '( ب ) الآن تورينغ',
 'source': 'ArabicMMLU',
 'document': 'ArabicMMLU_Group_27_003',
 'length': 5,
 'split': 'test',
 'has_sent_end_punct': False,
 'labels': [0, 0, 0, 0, 1],
 'ends_with_sent_end_punct': False,
 'num_common_punct': 0,
 'num_form_punct': 2,
 'ends_with_comma': False,
 'starts_with_conjunction_word': False,
 'ends_with_pnx': False,
 'ends_with_.': False,
 'num_.': 0,
 'ends_with_,': False,
 'num_,': 0,
 'ends_with_؛': False,
 'num_؛': 0,
 'ends_with_!': False,
 'num_!': 0,
 'ends_with_؟': False,
 'num_؟': 0}

In [14]:
src_total_sents

defaultdict(<function __main__.<lambda>()>,
            {'train': defaultdict(int,
                         {'ArabicMMLU': 25693,
                          'Diwan': 571207,
                          'Spacetoon Songs': 855,
                          'My Language Sings': 424,
                          'Hanada Final Poems': 388,
                          'Kashkul': 404,
                          'Kalima': 2370,
                          'Green Library': 2566,
                          'Hindawi': 20874,
                          'Majed': 9679,
                          'Curriculum': 13432,
                          'Wikipedia': 5156,
                          'Sara': 1537,
                          'Old Testament': 2185,
                          'Hanging Odes': 544,
                          'Quran': 4402,
                          'ZAEBUC': 770,
                          'ALC': 581,
                          'Arabian Nights': 824,
                          'Hadith': 809,
                

In [28]:
for split in data:
    for ex in data[split]:
        if ex["source"] == "al-Kashkuul":
            print(ex["sent"])
            print(ex["labels"])
            break


كنت أحس بألم دائما لأنه ليس لدى أطفالنا كتاب مختارات أدبية ,
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]


In [15]:
sent_lvl_stats_pnx = defaultdict(lambda: defaultdict(int))
sent_lvl_stats_non_pnx = defaultdict(lambda: defaultdict(int))

for split in data:
    for ex in data[split]:
        if ex["ends_with_pnx"]:
            sent_lvl_stats_pnx[split][ex["source"]] += 1
        else:
            sent_lvl_stats_non_pnx[split][ex["source"]] += 1

for split in data:
    for src in all_sources[split]:
        assert sent_lvl_stats_pnx[split][src] + sent_lvl_stats_non_pnx[split][src] == src_total_sents[split][src]
print(f"Everything adds up!")

rows = []
for split in data:
    for src in all_sources[split]:
        rows.append({
            "source": src,
            "split": split,
            "num_docs": n_docs_per_src[split][src],
            "n_sent_end_w_pnx": sent_lvl_stats_pnx[split][src],
            "n_sent_end_wo_pnx": sent_lvl_stats_non_pnx[split][src],
            "n_sent_total": src_total_sents[split][src],
            
        })

sent_lvl_stats_df = pd.DataFrame(rows)
os.makedirs("stats", exist_ok=True)
sent_lvl_stats_df.to_csv("stats/sent_lvl_stats.tsv", sep="\t", index=False)
sent_lvl_stats_df.head()

Everything adds up!


,source,split,num_docs,n_sent_end_w_pnx,n_sent_end_wo_pnx,n_sent_total
0,Quran,train,74,4402,0,4402
1,Green Library,train,52,2314,252,2566
2,WikiNews,train,47,599,96,695
3,Curriculum,train,91,8962,4470,13432
4,Kashkul,train,15,146,258,404


In [16]:
# word level stats
word_lvl_stats_seg = defaultdict(lambda: defaultdict(int))
word_lvl_stats_non_seg = defaultdict(lambda: defaultdict(int))
word_lvl_stats_pnx_seg = defaultdict(lambda: defaultdict(int))
word_lvl_stats_pnx_non_seg = defaultdict(lambda: defaultdict(int))
total_words = defaultdict(lambda: defaultdict(int))
total_pnx_words = defaultdict(lambda: defaultdict(int))
total_non_pnx_words = defaultdict(lambda: defaultdict(int))
def is_pnx(word):
    pnx_pattern = re.compile(r'[^\w\s]')
    return re.search(pnx_pattern, word) is not None

for split in data:
    for ex in data[split]:
        words = ex["sent"].split(" ")
        total_words[split][ex["source"]] += len(words)
        for i, word in enumerate(words):
            label = ex["labels"][i]
            if is_pnx(word):
                total_pnx_words[split][ex["source"]] += 1
                if label == 1:
                    word_lvl_stats_pnx_seg[split][ex["source"]] += 1
                else:
                    word_lvl_stats_pnx_non_seg[split][ex["source"]] += 1
            else:
                total_non_pnx_words[split][ex["source"]] += 1
                if label == 1:
                    word_lvl_stats_seg[split][ex["source"]] += 1
                else:
                    word_lvl_stats_non_seg[split][ex["source"]] += 1

# check if everything adds up
for split in data:
    for src in all_sources[split]:
        assert word_lvl_stats_pnx_seg[split][src] + word_lvl_stats_pnx_non_seg[split][src] + word_lvl_stats_seg[split][src] + word_lvl_stats_non_seg[split][src] == total_words[split][src], f"Total words for {src} in {split} are not equal to the sum of pnx and non-pnx words, {word_lvl_stats_pnx_seg[split][src]} + {word_lvl_stats_pnx_non_seg[split][src]} + {word_lvl_stats_seg[split][src]} + {word_lvl_stats_non_seg[split][src]} != {total_words[split][src]}"
        assert total_pnx_words[split][src] + total_non_pnx_words[split][src] == total_words[split][src], f"Total words for {src} in {split} are not equal to the sum of pnx and non-pnx words, {total_pnx_words[split][src]} + {total_non_pnx_words[split][src]} != {total_words[split][src]}"
        assert word_lvl_stats_pnx_seg[split][src] + word_lvl_stats_pnx_non_seg[split][src] == total_pnx_words[split][src], f"Total pnx words for {src} in {split} are not equal to the sum of pnx and non-pnx words, {word_lvl_stats_pnx_seg[split][src]} + {word_lvl_stats_pnx_non_seg[split][src]} != {total_pnx_words[split][src]}"

rows = []
for split in data:
    for src in all_sources[split]:
        rows.append({
            "source": src,
            "split": split,
            "total_words": total_non_pnx_words[split][src],
            "total_pnx": total_pnx_words[split][src],
            "words_seg": word_lvl_stats_seg[split][src],
            "pnx_seg": word_lvl_stats_pnx_seg[split][src],
            "words_no_seg": word_lvl_stats_non_seg[split][src],
            "pnx_no_seg": word_lvl_stats_pnx_non_seg[split][src]
        })

word_lvl_stats_df = pd.DataFrame(rows)
word_lvl_stats_df.to_csv("stats/word_lvl_stats.tsv", sep="\t", index=False)
word_lvl_stats_df.head()


,source,split,total_words,total_pnx,words_seg,pnx_seg,words_no_seg,pnx_no_seg
0,Quran,train,60085,8804,0,4402,60085,4402
1,Green Library,train,31708,9193,252,2314,31456,6879
2,WikiNews,train,11746,1232,96,599,11650,633
3,Curriculum,train,99977,19261,4470,8962,95507,10299
4,Kashkul,train,2241,307,258,146,1983,161


In [17]:
from collections import defaultdict
import pandas as pd

# ---------- CONFIG: adjust if your example keys differ ----------
TOK_KEY = "sent"     # token list per example
LAB_KEY = "labels"   # label list per example (1 == EOS)
# ---------------------------------------------------------------

def count_dotplus_spans(tokens, labels):
    """
    Returns:
      total_spans: number of runs of '.' of length>=2
      eos_spans:   number of those runs that contain at least one label==1
    """
    total_spans = 0
    eos_spans = 0
    i = 0
    n = len(tokens)

    while i < n:
        if tokens[i] == ".":
            j = i
            while j < n and tokens[j] == ".":
                j += 1
            run_len = j - i
            if run_len >= 2:
                total_spans += 1
                if any(l == 1 for l in labels[i:j]):
                    eos_spans += 1
            i = j
        else:
            i += 1
    return total_spans, eos_spans


# -----------------------------
# 1) Aggregate ".+" counts by (split, source)
# -----------------------------
dotplus_total = defaultdict(lambda: defaultdict(int))
dotplus_eos   = defaultdict(lambda: defaultdict(int))

for split in data:
    for ex in data[split]:
        src = ex["source"]
        tokens = ex[TOK_KEY].split(" ")
        labels = ex[LAB_KEY]
        t, e = count_dotplus_spans(tokens, labels)
        # print(f"Total spans: {t}, EOS spans: {e} for example {ex['id']}")
        dotplus_total[split][src] += t
        dotplus_eos[split][src] += e


# -----------------------------
# 2) Your punctuation stats (kept, but EOS counting fixed)
# -----------------------------
pnx_type_stats_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
pnx_type_stats_eos_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
pnx_type_stats_non_eos_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

for split in data:
    for ex in data[split]:
        src = ex["source"]
        for pnx in sent_end_punct:
            n = ex.get(f"num_{pnx}", 0)
            if n > 0:
                pnx_type_stats_cnt[split][src][pnx] += n
                if ex.get(f"ends_with_{pnx}", False):
                    # count exactly 1 EOS occurrence for this punctuation type in this example
                    pnx_type_stats_eos_cnt[split][src][pnx] += 1
                    pnx_type_stats_non_eos_cnt[split][src][pnx] += (n - 1)
                else:
                    pnx_type_stats_non_eos_cnt[split][src][pnx] += n

# sanity check
for split in data:
    for src in all_sources[split]:
        for pnx in sent_end_punct:
            assert pnx_type_stats_cnt[split][src][pnx] == (
                pnx_type_stats_eos_cnt[split][src][pnx] + pnx_type_stats_non_eos_cnt[split][src][pnx]
            ), (
                f"Mismatch for split={split} src={src} pnx={pnx}: "
                f"{pnx_type_stats_cnt[split][src][pnx]} != "
                f"{pnx_type_stats_eos_cnt[split][src][pnx]} + {pnx_type_stats_non_eos_cnt[split][src][pnx]}"
            )

dotplus_total = defaultdict(lambda: defaultdict(int))
dotplus_eos = defaultdict(lambda: defaultdict(int))
dotplus_sum_periods = defaultdict(lambda: defaultdict(int))  # NEW

for split in data:
    for ex in data[split]:
        src = ex["source"]
        tokens = ex["sent"].split(" ")
        labels = ex["labels"]

        i = 0
        while i < len(tokens):
            if tokens[i] == ".":
                j = i
                while j < len(tokens) and tokens[j] == ".":
                    j += 1
                run_len = j - i
                if run_len >= 2:
                    dotplus_total[split][src] += 1
                    dotplus_sum_periods[split][src] += run_len
                    if any(l == 1 for l in labels[i:j]):
                        dotplus_eos[split][src] += 1
                i = j
            else:
                i += 1

# -----------------------------
# 3) Build rows + append ".+" row per (split, source)
# -----------------------------
rows = []
for split in data:
    for src in all_sources[split]:

        # values from span stats
        num_spans = dotplus_total[split][src]                  # number of ".+" spans
        num_spans_eos = dotplus_eos[split][src]                # spans that contain at least one label==1
        sum_span_periods = dotplus_sum_periods[split][src]     # total '.' tokens inside those spans

        # how many '.' tokens to remove from the "." totals
        # (replace each run_len by 1 => remove run_len-1 per span)
        dots_to_remove = sum_span_periods #- num_spans

        # how many EOS '.' tokens to remove from "." EOS totals
        # (move exactly 1 EOS credit per span-with-eos from '.' to '.+')
        eos_to_remove = num_spans_eos

        for pnx in sent_end_punct:
            total = pnx_type_stats_cnt[split][src][pnx]
            seg = pnx_type_stats_eos_cnt[split][src][pnx]
            noseg = pnx_type_stats_non_eos_cnt[split][src][pnx]

            # ✅ adjust "." row only
            if pnx == ".":
                total = total - dots_to_remove
                seg = seg - eos_to_remove
                noseg = total - seg

                # safety clamps (avoid negatives if upstream heuristics differ)
                if total < 0: total = 0
                if seg < 0: seg = 0
                if seg > total: seg = total
                noseg = total - seg

            rows.append({
                "source": src,
                "split": split,
                "pnx": f"<s>{pnx}</s>",
                "seg": seg,
                "no seg": noseg,
                "total": total,
            })

        # ".+" row (span-based)
        rows.append({
            "source": src,
            "split": split,
            "pnx": "<s>.+</s>",
            "seg": num_spans_eos,
            "no seg": num_spans - num_spans_eos,
            "total": num_spans,
        })


pnx_type_stats_df = pd.DataFrame(rows)
pnx_type_stats_df.to_csv("stats/pnx_type_stats_1.tsv", sep="\t", index=False)
pnx_type_stats_df.head()


,source,split,pnx,seg,no seg,total
0,Quran,train,<s>.</s>,0,0,0
1,Quran,train,"<s>,</s>",0,0,0
2,Quran,train,<s>؛</s>,0,0,0
3,Quran,train,<s>!</s>,0,0,0
4,Quran,train,<s>؟</s>,0,0,0


In [18]:
# sent end pnx lvl stats
pnx_type_stats_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
pnx_type_stats_eos_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))
pnx_type_stats_non_eos_cnt = defaultdict(lambda: defaultdict(lambda: defaultdict(int)))

for split in data:
    for ex in data[split]:
        for pnx in sent_end_punct:
            if ex[f"num_{pnx}"] > 0:
                pnx_type_stats_cnt[split][ex["source"]][pnx] += ex[f"num_{pnx}"]
                
                if ex[f"ends_with_{pnx}"]:
                    pnx_type_stats_eos_cnt[split][ex["source"]][pnx] += 1
                    pnx_type_stats_eos_cnt[split][ex["source"]][pnx] += ex[f"num_{pnx}"] - 1
                else:
                    pnx_type_stats_non_eos_cnt[split][ex["source"]][pnx] += ex[f"num_{pnx}"]
for split in data:
    for src in all_sources[split]:
        for pnx in sent_end_punct:
            assert pnx_type_stats_cnt[split][src][pnx] == pnx_type_stats_eos_cnt[split][src][pnx] + pnx_type_stats_non_eos_cnt[split][src][pnx], f"Total pnx count for {src} in {split} are not equal to the sum of {pnx} and non-pnx words, {pnx_type_stats_cnt[split][src][pnx]} != {pnx_type_stats_eos_cnt[split][src][pnx]} + {pnx_type_stats_non_eos_cnt[split][src][pnx]}"


rows = []
for split in data:
    for src in all_sources[split]:
        for pnx in sent_end_punct:
            rows.append({
                "source": src,
                "split": split,
                "pnx":f"<s>{pnx}</s>",
                "seg": pnx_type_stats_eos_cnt[split][src][pnx],
                "no seg": pnx_type_stats_non_eos_cnt[split][src][pnx],
                "total": pnx_type_stats_cnt[split][src][pnx],
            })

pnx_type_stats_df = pd.DataFrame(rows)
pnx_type_stats_df.to_csv("stats/pnx_type_stats.tsv", sep="\t", index=False)
pnx_type_stats_df.head()
            

,source,split,pnx,seg,no seg,total
0,Quran,train,<s>.</s>,0,0,0
1,Quran,train,"<s>,</s>",0,0,0
2,Quran,train,<s>؛</s>,0,0,0
3,Quran,train,<s>!</s>,0,0,0
4,Quran,train,<s>؟</s>,0,0,0


In [19]:
doc_lvl_sent_cnts = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int)))) # split -> src -> pnx -> mean, max, min, std
doc_lvl_pnx_nums = defaultdict(lambda: defaultdict(lambda: defaultdict(lambda: defaultdict(int)))) # split -> src -> doc -> pnx -> cnt
# ex: doc_lvl_pnx_nums["train"]["Quran"]["Doc1"]["cnt_."] = 
for split in data:
    for ex in data[split]:
        doc_id = ex["document"]
        for pnx in sent_end_punct:
            if ex[f"num_{pnx}"] > 0:
                # print(doc_lvl_pnx_nums[split][ex["source"]][doc_id][f"cnt_{pnx}"])
                doc_lvl_pnx_nums[split][ex["source"]][doc_id][f"cnt_{pnx}"] += ex[f"num_{pnx}"]
                doc_lvl_pnx_nums[split][ex["source"]][doc_id][f"cnt_eos_{pnx}"] += ex[f"num_{pnx}"] if ex[f"ends_with_{pnx}"] else 0
        # print(doc_lvl_sent_cnts[split][ex["source"]][doc_id])
        doc_lvl_sent_cnts[split][ex["source"]][doc_id]["cnt_num_sent"] += 1
        # doc_lvl_sent_cnts[split][ex["source"]][doc_id]
        doc_lvl_sent_cnts[split][ex["source"]][doc_id]["cnt_pnx_eos_sent"] += ex[f"ends_with_sent_end_punct"]
        
    
doc_lvl_stats = defaultdict(lambda: defaultdict(lambda: defaultdict(int))) # split -> src -> mean, std, max, min

for split in doc_lvl_sent_cnts:
    for src in doc_lvl_sent_cnts[split]:
        for pnx in sent_end_punct:
            pnx_num = []
            pnx_eos_num = []
            for doc_id in doc_lvl_pnx_nums[split][src]:
                pnx_num.append(doc_lvl_pnx_nums[split][src][doc_id][f"cnt_{pnx}"])
                pnx_eos_num.append(doc_lvl_pnx_nums[split][src][doc_id][f"cnt_eos_{pnx}"])
            if sum(pnx_num) > 0:
                doc_lvl_stats[split][src][f"{pnx}_stats"] = {
                    "mean": np.mean(pnx_num),
                    "std": np.std(pnx_num),
                    "max": np.max(pnx_num),
                    "min": np.min(pnx_num)
                }
                doc_lvl_stats[split][src][f"{pnx}_eos_stats"] = {
                    "mean": np.mean(pnx_eos_num),
                    "std": np.std(pnx_eos_num),
                    "max": np.max(pnx_eos_num),
                    "min": np.min(pnx_eos_num)
                }

rows = []
for split in doc_lvl_stats:
    for src in doc_lvl_stats[split]:
        for pnx in sent_end_punct:
            rows.append({
                "source": src,
                "split": split,
                "pnx_type": f"<s>{pnx}</s>",
                
                "pnx_type_mean": doc_lvl_stats[split][src][f"{pnx}_stats"].get("mean", 0) if f"{pnx}_stats" in doc_lvl_stats[split][src] else 0,
                "pnx_type_std": doc_lvl_stats[split][src][f"{pnx}_stats"].get("std", 0) if f"{pnx}_stats" in doc_lvl_stats[split][src] else 0,
                "pnx_type_eos_mean": doc_lvl_stats[split][src][f"{pnx}_eos_stats"].get("mean", 0) if f"{pnx}_eos_stats" in doc_lvl_stats[split][src] else 0,
                "pnx_type_eos_std": doc_lvl_stats[split][src][f"{pnx}_eos_stats"].get("std", 0) if f"{pnx}_eos_stats" in doc_lvl_stats[split][src] else 0,
            })

doc_lvl_stats_df = pd.DataFrame(rows)
doc_lvl_stats_df.to_csv("stats/doc_lvl_stats.tsv", sep="\t", index=False)
doc_lvl_stats_df.head()


,source,split,pnx_type,pnx_type_mean,pnx_type_std,pnx_type_eos_mean,pnx_type_eos_std
0,ArabicMMLU,train,<s>.</s>,28.625000,33.151580,22.939286,30.723419
1,ArabicMMLU,train,"<s>,</s>",2.357143,4.277134,0.010714,0.133200
2,ArabicMMLU,train,<s>؛</s>,0.107143,0.417414,0.025000,0.196623
3,ArabicMMLU,train,<s>!</s>,0.021429,0.222578,0.007143,0.119309
4,ArabicMMLU,train,<s>؟</s>,4.800000,7.304793,4.492857,6.946218


In [ ]:
source_wise_stats_pnx = {split: defaultdict(int) for split in ["train", "dev", "test", "STask"]}
source_wise_stats_sem = {split: defaultdict(int) for split in ["train", "dev", "test", "STask"]}
for split in data.keys():
    for example in data[split]:
        if example["ends_with_sent_end_punct"]:
            source_wise_stats_pnx[split][example["source"]] += 1
        else:
            source_wise_stats_sem[split][example["source"]] += 1
            
rows = []

for split in ["train", "dev", "test", "STask"]:
    domains = (set(source_wise_stats_pnx[split].keys()) | set(source_wise_stats_sem[split].keys()))
    for domain in domains:
        rows.append({
            "source": domain,
            "split": split,
            "pnx_cnt": source_wise_stats_pnx[split].get(domain, 0),
            "sem_cnt": source_wise_stats_sem[split].get(domain, 0)
        })

src_wise_stats_df = pd.DataFrame(rows).set_index(["source", "split"])
src_wise_stats_df["Total"] = src_wise_stats_df["pnx_cnt"] + src_wise_stats_df["sem_cnt"]
src_wise_stats_df["pnx_ratio"] = src_wise_stats_df["pnx_cnt"] / src_wise_stats_df["Total"]
src_wise_stats_df["sem_ratio"] = src_wise_stats_df["sem_cnt"] / src_wise_stats_df["Total"]
src_wise_stats_df


,,pnx_cnt,sem_cnt,Total,pnx_ratio,sem_ratio
source,split,,,,,
Quran,train,0,4402,4402,0.000000,1.000000
Green Library,train,2134,432,2566,0.831645,0.168355
WikiNews,train,580,115,695,0.834532,0.165468
Curriculum,train,5940,7492,13432,0.442228,0.557772
Kashkul,train,98,306,404,0.242574,0.757426
...,...,...,...,...,...,...
Hindawi,STask,881,265,1146,0.768761,0.231239
Wikipedia,STask,748,76,824,0.907767,0.092233
Curriculum,STask,428,439,867,0.493656,0.506344


In [6]:
all_data = []
for split in data.keys():
    all_data.extend(data[split])


In [7]:
train_data = []
dev_data = []
test_data = []
for split in data.keys():
    for example in data[split]:
        if split == "train":
            train_data.append(example)
        elif split == "dev":
            dev_data.append(example)
        else:
            test_data.append(example)

In [8]:
train_data[0]

{'id': 10100290001,
 'sent': 'مجلة كل الأولاد وكل البنات',
 'domain': 'Majed',
 'document': 'BAREC_Majed_0413_1987_001.txt',
 'length': 5,
 'Annotator': 'A2',
 'R3': 1,
 'R5': 1,
 'R7': 2,
 'R19': 7,
 'num_sent_end_punct': 0,
 'has_sent_end_punct': False,
 'split': 'train',
 'ends_with_sent_end_punct': False,
 'num_common_punct': 0,
 'num_form_punct': 0,
 'ends_with_comma': False,
 'starts_with_conjunction_word': False}

In [9]:
len(dev_data), len(test_data), len(train_data)

(7310, 7286, 54845)

In [47]:
eos_wide = df.pivot_table(
    index=["source", "split"],
    columns="pnx",
    values="pnx_eos_cnt",
    aggfunc="sum",
    fill_value=0
)
sem_eos_wide = df.pivot_table(
    index=["source", "split"],
    columns="pnx",
    values="pnx_non_eos_cnt",
    aggfunc="sum",
    fill_value=0
)
eos_wide.columns = [f"{c}_eos_cnts" for c in eos_wide.columns]
sem_eos_wide.columns = [f"{c}_sem_cnts" for c in sem_eos_wide.columns]
pivoted_df = pd.concat([eos_wide, sem_eos_wide], axis=1)
new_df = src_wise_stats_df.copy()
new_df = new_df.join(pivoted_df, how="left").fillna(0)
final_df = new_df.reset_index(level="split")
os.makedirs(f"./stats/", exist_ok=True)
new_df.to_csv(f"./stats/src_wise_stats.tsv", sep="\t")

In [45]:
final_df = new_df.reset_index(level="split")
final_df

,split,pnx_cnt,sem_cnt,Total,pnx_ratio,sem_ratio
source,,,,,,
New Testament,train,367,35,402,0.912935,0.087065
Poems and News,train,23,368,391,0.058824,0.941176
al-Kashkuul,train,100,230,330,0.303030,0.696970
ReadMe++,train,684,369,1053,0.649573,0.350427
Emarati Curriculum,train,4899,4872,9771,0.501382,0.498618
...,...,...,...,...,...,...
Mama Makes Bread,test,21,18,39,0.538462,0.461538
Old Testament,test,51,3,54,0.944444,0.055556
chatGPT,test,152,0,152,1.000000,0.000000


In [ ]:
src_wise_stats_df.join()

In [21]:
split_data["train"]["eos_cnts"]

{'Majed': {'<s>.</s>': 3972,
  '<s>,</s>': 1226,
  '<s>!</s>': 1088,
  '<s>؟</s>': 551,
  '<s>:</s>': 75},
 'ArabicMMLU': {'<s>.</s>': 289},
 'Hindawi': {'<s>.</s>': 5408,
  '<s>,</s>': 2077,
  '<s>؛</s>': 594,
  '<s>!</s>': 288,
  '<s>؟</s>': 492,
  '<s>:</s>': 98},
 'Emarati Curriculum': {'<s>.</s>': 3503,
  '<s>,</s>': 740,
  '<s>؛</s>': 54,
  '<s>!</s>': 36,
  '<s>؟</s>': 566,
  '<s>:</s>': 1462},
 'Green Library': {'<s>.</s>': 1390,
  '<s>,</s>': 419,
  '<s>؛</s>': 49,
  '<s>!</s>': 147,
  '<s>؟</s>': 112,
  '<s>:</s>': 32},
 'Spacetoon Songs': {'<s>.</s>': 149, '<s>,</s>': 22, '<s>؟</s>': 2},
 'chatGPT': {'<s>.</s>': 197},
 'My Language Sings': {'<s>.</s>': 72,
  '<s>,</s>': 6,
  '<s>!</s>': 8,
  '<s>؟</s>': 29,
  '<s>:</s>': 8},
 'Poems and News': {'<s>.</s>': 12, '<s>؟</s>': 11},
 'Wikipedia': {'<s>.</s>': 3037,
  '<s>,</s>': 895,
  '<s>؛</s>': 90,
  '<s>؟</s>': 15,
  '<s>:</s>': 37},
 'ReadMe++': {'<s>.</s>': 667,
  '<s>؛</s>': 1,
  '<s>!</s>': 11,
  '<s>؟</s>': 5,
  '<s>:</s>

In [16]:
def get_num_eos_punct_per_col(col, punct, data, rotate=True):
    count_eos_punct_per_col = defaultdict(int)
    count_punct_per_col = defaultdict(int)
    for example in data:
        if example["sent"].split(" ")[-1] == punct:
            count_eos_punct_per_col[example[col]] += 1
        count_punct_per_col[example[col]] += example["sent"].count(punct)
    # plt.figure(figsize=(10, 6))
    # plt.title(f"Count of EOS {punct} per {col}")
    # plt.bar(count_punct_per_col.keys(), count_punct_per_col.values(), alpha=0.5)
    # plt.bar(count_eos_punct_per_col.keys(), count_eos_punct_per_col.values())
    # plt.xlabel(col)
    # plt.ylabel("Count of EOS Punct")
    # if rotate:
    #     plt.xticks(rotation=90)
    # # plt.tight_layout()
    # plt.grid(True)
    # plt.legend()
    # plt.show()
    return count_eos_punct_per_col, count_punct_per_col

# period_eos, period_punct = get_num_eos_punct_per_col("domain", ".", all_data)
# print(f"="*100)
# comma_eos, comma_punct = get_num_eos_punct_per_col("domain", ",", all_data)
# print(f"="*100)
# open_bracket_eos, open_bracket_punct = get_num_eos_punct_per_col("domain", "(", all_data)
# print(f"="*100)
# close_bracket_eos, close_bracket_punct = get_num_eos_punct_per_col("domain", ")", all_data)
# print(f"="*100)
# semicolon_eos, semicolon_punct = get_num_eos_punct_per_col("domain", ";", all_data)
# print(f"="*100)
# semicolon_eos, semicolon_punct = get_num_eos_punct_per_col("domain", "؛", all_data)
# print(f"="*100)
# exclamation_eos, exclamation_punct = get_num_eos_punct_per_col("domain", "!", all_data)
# print(f"="*100)
# question_mark_eos, question_mark_punct = get_num_eos_punct_per_col("domain", "؟", all_data)
# print(f"="*100)
# question_mark_eos, question_mark_punct = get_num_eos_punct_per_col("domain", "?", all_data)
# print(f"="*100)
# colon_eos, colon_punct = get_num_eos_punct_per_col("domain", ":", all_data)
# print(f"="*100)

# all_puncts_used = [".", ",", "(", ")", ";", "؛", "!", "؟", "?", ":", "-", '"', "'"]
split_data = {}
for split in ["train", "dev", "test"]:
    sent_end_punct = [ ".", ",", "،", ";", "؛", "!", "?", "؟", ":"]
    all_punct_cnts_eos_per_source = {}
    all_punct_cnts_punct_per_source = {}
    if split == "train":
        data_to_use = train_data
    elif split == "dev":
        data_to_use = dev_data
    else:
        data_to_use = test_data
    for punct in sent_end_punct:
        cnt_eos, cnt_punct = get_num_eos_punct_per_col("domain", punct, data_to_use)
        for source in cnt_eos.keys():
            if source not in all_punct_cnts_eos_per_source:
                all_punct_cnts_eos_per_source[source] = {}
            all_punct_cnts_eos_per_source[source][f"<s>{punct}</s>"] = cnt_eos[source]
            if source not in all_punct_cnts_punct_per_source:
                all_punct_cnts_punct_per_source[source] = {}
            all_punct_cnts_punct_per_source[source][f"<s>{punct}</s>"] = cnt_punct[source]
    split_data[split] = {
        "eos_cnts": all_punct_cnts_eos_per_source,
        "punct_cnts": all_punct_cnts_punct_per_source
    }

In [17]:
all_punct_cnts_punct_per_source

{'Majed': {'<s>.</s>': 452,
  '<s>,</s>': 579,
  '<s>!</s>': 16,
  '<s>؟</s>': 67,
  '<s>:</s>': 88},
 'Hindawi': {'<s>.</s>': 644,
  '<s>,</s>': 1037,
  '<s>؛</s>': 133,
  '<s>!</s>': 89,
  '<s>؟</s>': 133,
  '<s>:</s>': 135},
 'Emarati Curriculum': {'<s>.</s>': 813,
  '<s>,</s>': 932,
  '<s>؛</s>': 82,
  '<s>!</s>': 5,
  '<s>؟</s>': 124,
  '<s>:</s>': 540},
 'Green Library': {'<s>.</s>': 206,
  '<s>,</s>': 352,
  '<s>؛</s>': 15,
  '<s>!</s>': 15,
  '<s>؟</s>': 9},
 'Spacetoon Songs': {'<s>.</s>': 6, '<s>,</s>': 9},
 'chatGPT': {'<s>.</s>': 152},
 'Mama Makes Bread': {'<s>.</s>': 32},
 'Wikipedia': {'<s>.</s>': 255,
  '<s>,</s>': 510,
  '<s>؛</s>': 8,
  '<s>:</s>': 40},
 'ReadMe++': {'<s>.</s>': 126, '<s>!</s>': 4, '<s>؟</s>': 5, '<s>:</s>': 7},
 'WikiNews': {'<s>.</s>': 74, '<s>,</s>': 66, '<s>!</s>': 1, '<s>؟</s>': 1},
 'Zayed Arabic-English Bilingual Undergraduate Corpus (ZAEBUC)': {'<s>.</s>': 90,
  '<s>,</s>': 78,
  '<s>؟</s>': 6},
 'Basic Travel Expressions Corpus (BTEC)': {'<s>

In [22]:
df_rows = []
# cols we want:
# pnx, pnx_eos_cnt, pnx_non_eos_cnt, pnx_total_cnt, source, split
for split in split_data.keys():
    # collect data for each pnx
    for source in split_data[split]["eos_cnts"].keys():
        for pnx in split_data[split]["eos_cnts"][source].keys():
            df_rows.append({
                "pnx": pnx,
                "pnx_eos_cnt": split_data[split]["eos_cnts"][source][pnx],
                "pnx_non_eos_cnt": split_data[split]["punct_cnts"][source][pnx] - split_data[split]["eos_cnts"][source][pnx],
                "pnx_total_cnt": split_data[split]["punct_cnts"][source][pnx],
                "source": source,
                "split": split
            })
df = pd.DataFrame(df_rows)
# sort by pnx
df = df.sort_values(by="pnx", ascending=False)


In [24]:
df["eos_cnts"] = df["pnx"].apply(lambda x: f"{pnx}_eos_cnts")
df["sem_cnts"] = df["pnx"].apply(lambda x: f"{pnx}_sem_cnts")

In [30]:
df.to_csv("pnx_only_data_stats.tsv", sep="\t", index=False)

In [26]:
from this import s


conjunction_words = ["و", "ثم", "ف"]

# for , get the number of times the first word of the next sentence starts with one of the conjunction words
# for each source, split
conj_cont_cnts = {"train": {}, "dev": {}, "test": {}}
for split in data:
    for ex1, ex2 in zip(data[split], data[split][1:]):
        if ex1["document"] != ex2["document"]:
            continue
        if not ex1["ends_with_comma"] and ex2["starts_with_conjunction_word"]:
            if ex1["domain"] not in conj_cont_cnts[split]:
                conj_cont_cnts[split][ex1["domain"]] = 0
            conj_cont_cnts[split][ex1["domain"]] += 1
            
df_conj_cont_cnts = pd.DataFrame(conj_cont_cnts).fillna(0)

In [27]:
df_conj_cont_cnts

,train,dev,test
Majed,1115.0,129.0,73.0
ArabicMMLU,115.0,17.0,14.0
Hindawi,2482.0,259.0,264.0
Emarati Curriculum,430.0,61.0,122.0
Green Library,745.0,28.0,62.0
Spacetoon Songs,50.0,2.0,0.0
My Language Sings,54.0,0.0,0.0
Poems and News,39.0,0.0,0.0
Wikipedia,1022.0,127.0,94.0
ReadMe++,175.0,28.0,29.0


In [17]:
data["dev"]

[{'id': 10100010001,
  'sent': 'مجلة كل الأولاد وكل البنات',
  'domain': 'Majed',
  'document': 'BAREC_Majed_0229_1983_001.txt',
  'length': 5,
  'Annotator': 'A2',
  'R3': 1,
  'R5': 1,
  'R7': 2,
  'R19': 7,
  'num_sent_end_punct': 0,
  'has_sent_end_punct': False,
  'split': 'dev',
  'ends_with_sent_end_punct': False,
  'num_common_punct': 0,
  'num_form_punct': 0,
  'ends_with_comma': False,
  'starts_with_conjunction_word': False},
 {'id': 10100010002,
  'sent': 'ماجد',
  'domain': 'Majed',
  'document': 'BAREC_Majed_0229_1983_001.txt',
  'length': 1,
  'Annotator': 'A2',
  'R3': 1,
  'R5': 1,
  'R7': 1,
  'R19': 1,
  'num_sent_end_punct': 0,
  'has_sent_end_punct': False,
  'split': 'dev',
  'ends_with_sent_end_punct': False,
  'num_common_punct': 0,
  'num_form_punct': 0,
  'ends_with_comma': False,
  'starts_with_conjunction_word': False},
 {'id': 10100010003,
  'sent': 'الاربعاء 13 يوليو 1983م',
  'domain': 'Majed',
  'document': 'BAREC_Majed_0229_1983_001.txt',
  'length': 4,